In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'google-generativeai>=0.8.0',
], check=True)


In [ ]:

import os
import json
import time
import threading
import uuid
import math
from pathlib import Path
from datetime import datetime

import yaml
import requests
import google.generativeai as genai
import google.api_core.exceptions
from shared.gemini_rate_limiter import GeminiRateLimiter
from shared.secrets import load_secrets
from shared.workflow_kernel import WorkflowKernel
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
SEEDS_PATH      = WORK_DIR / 'seed_dialogues.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p3a.json'
CONFIG_DIR      = Path('/kaggle/input/S2S-pipline-v2-0-2/config')

SEEDS_PER_DOMAIN = 500
BATCH_SIZE       = 5
MAX_RETRIES      = 6
SAVE_EVERY       = 10
DOMAINS          = ['restaurant', 'banking', 'healthcare', 'education']
DIFFICULTY_DIST  = {'simple': 0.60, 'multi_tool': 0.30, 'escalation': 0.10}

In [ ]:
SECRETS = load_secrets(require_gemini=True)
HF_TOKEN         = SECRETS['HF_TOKEN_PRIMARY']
GEMINI_KEY = SECRETS.get('GEMINI_API_KEY_01') or SECRETS.get('GEMINI_API_KEY_02') or ''
genai.configure(api_key=GEMINI_KEY)
GEMINI_MODEL = genai.GenerativeModel("gemini-2.5-flash")
RATE_LIMITER = GeminiRateLimiter(rpm_limit=14)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'intent_taxonomy.yaml') as f:
    taxonomy = yaml.safe_load(f)
with open(CONFIG_DIR / 'tool_registry.yaml') as f:
    tool_registry = yaml.safe_load(f)

STAGE2_REPO = repos_cfg['repos']['stage2_moe']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage2 repo: {STAGE2_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — generated={state["stats"]["generated"]}')
            return state
        except Exception:
            pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE2_REPO}/resolve/main/checkpoint_p3a.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — generated={state["stats"]["generated"]}')
            return state
    except Exception:
        pass
    print('[checkpoint] fresh start')
    return {
        'by_domain': {d: 0 for d in DOMAINS},
        'stats': {'generated': 0, 'failed': 0, 'invalid': 0},
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p3a.json',
                repo_id=STAGE2_REPO,
                repo_type='dataset',
                commit_message='p3a checkpoint',
            )
            return
        except Exception:
            time.sleep(min(2 ** attempt, 60))


state = load_checkpoint()

In [ ]:
def build_system_prompt(domain):
    intents = taxonomy.get(domain, []) + taxonomy.get('cross_domain', [])
    tools   = list(tool_registry.get(domain, {}).keys())
    return f"""You are a dialogue generation system for Pakistani Urdu customer support AI training.
Generate ONLY a valid JSON array of dialogues — no preamble, no markdown, no explanation.

Domain: {domain}
Available intents: {json.dumps(intents)}
Available tools: {json.dumps(tools)}

Each dialogue object must have exactly these fields:
- id: string (unique)
- domain: \"{domain}\"
- difficulty: one of [simple, multi_tool, escalation]
- turns: array of turn objects
- intents_covered: array of intent strings
- tools_referenced: array of tool names
- has_code_switch: boolean

Each turn object must have:
- turn_id: integer starting from 0
- speaker: \"user\" or \"agent\"
- text: dialogue text in Urdu script naturally mixed with English
- intent: intent class string for user turns, null for agent turns
- action_type: \"speak\", \"tool_call\", or \"resolve\"
- tool_name: string or null
- tool_args: object or null
- tool_result: object or null

Rules:
- 3 to 8 turns per dialogue
- At least 1 tool_call turn per dialogue
- Mix Urdu and English naturally like real Pakistanis speak
- simple: 1 tool, direct resolution
- multi_tool: 2+ tools, chained
- escalation: tool cannot resolve, human handoff
- Tool args and results must be internally consistent"""


def generate_seed_batch(domain, difficulty, count, system_prompt):
    user_msg = f'Generate exactly {count} {difficulty} difficulty dialogues for the {domain} domain. Return only the JSON array.'
    for attempt in range(MAX_RETRIES):
        try:
            RATE_LIMITER.acquire()
            response = GEMINI_MODEL.generate_content(
                [system_prompt + "\n\n" + user_msg],
                generation_config={"response_mime_type": "application/json"}
            )
            raw = response.text.strip().replace('```json', '').replace('```', '').strip()
            dialogues = json.loads(raw)
            if not isinstance(dialogues, list):
                raise ValueError('Expected JSON array')
            return dialogues
        except json.JSONDecodeError as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  [generate] JSON parse failed: {e}')
                return []
            time.sleep(5 * (attempt + 1))
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  [generate] API failed: {e}')
                return []
            time.sleep(min(2 ** attempt, 60))
    return []


def validate_dialogue(d):
    required = ['id', 'domain', 'difficulty', 'turns', 'intents_covered', 'tools_referenced', 'has_code_switch']
    if not all(k in d for k in required):
        return False
    if not isinstance(d['turns'], list) or len(d['turns']) < 2:
        return False
    if not any(t.get('action_type') == 'tool_call' for t in d['turns']):
        return False
    return True

In [ ]:
for domain in DOMAINS:
    already_done = state['by_domain'].get(domain, 0)
    remaining    = SEEDS_PER_DOMAIN - already_done

    if remaining <= 0:
        print(f'[{domain}] already complete ({already_done}/{SEEDS_PER_DOMAIN})')
        continue

    print(f'\n[{domain}] generating {remaining} more seeds ({already_done} already done)')
    system_prompt = build_system_prompt(domain)

    diff_counts = {
        'simple':     math.ceil(remaining * DIFFICULTY_DIST['simple']),
        'multi_tool': math.ceil(remaining * DIFFICULTY_DIST['multi_tool']),
        'escalation': math.ceil(remaining * DIFFICULTY_DIST['escalation']),
    }

    for difficulty, total_for_diff in diff_counts.items():
        generated_this_diff = 0
        batch_num           = 0

        while generated_this_diff < total_for_diff:
            ask_count  = min(BATCH_SIZE, total_for_diff - generated_this_diff)
            dialogues  = generate_seed_batch(domain, difficulty, ask_count, system_prompt)
            batch_num += 1

            valid_count = 0
            for d in dialogues:
                if not validate_dialogue(d):
                    with cp_lock:
                        state['stats']['invalid'] += 1
                    continue
                d['id']           = f'{domain}_seed_{str(uuid.uuid4())[:8]}'
                d['generated_at'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
                d['split']        = 'train'
                with open(SEEDS_PATH, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(d, ensure_ascii=False) + '\n')
                valid_count         += 1
                generated_this_diff += 1
                with cp_lock:
                    state['stats']['generated']  += 1
                    state['by_domain'][domain]   += 1

            if not dialogues:
                with cp_lock:
                    state['stats']['failed'] += 1

            if batch_num % SAVE_EVERY == 0:
                save_checkpoint(state, upload=True)

            print(f'  [{domain}/{difficulty}] batch {batch_num} valid={valid_count}/{len(dialogues)} total={generated_this_diff}/{total_for_diff}')
            time.sleep(1)

    save_checkpoint(state, upload=True)

total_seeds = sum(1 for _ in open(SEEDS_PATH, encoding='utf-8')) if SEEDS_PATH.exists() else 0
print(f'\n[p3a] {total_seeds} seed dialogues written')
print('[done] ready for p3b_augment.ipynb')